In [1]:
import os, sys, glob

os.environ.setdefault("SPARK_HOME", "/usr/local/spark")
spark_home = os.environ["SPARK_HOME"]

sys.path.insert(0, os.path.join(spark_home, "python"))
py4j_zips = glob.glob(os.path.join(spark_home, "python", "lib", "py4j-*.zip"))
if py4j_zips:
    sys.path.insert(0, py4j_zips[0])

os.environ["PYSPARK_PYTHON"] = sys.executable

import pyspark
print("pyspark version:", pyspark.__version__)


pyspark version: 3.5.0


In [3]:
!pip install -q kafka-python psycopg2-binary

In [5]:
import json, time, uuid
from kafka import KafkaProducer

# "kafka" is the container name from docker-compose.yml, 29092 is the
# internal-network port (Jupyter talks to Kafka over the Docker network,
# not via localhost).
KAFKA_BOOTSTRAP = "kafka:29092"
TOPIC = "upi.transactions"  # the Kafka topic we're testing

# A producer sends messages to Kafka. value_serializer tells it how to
# turn our Python dict into bytes before sending: dict -> JSON string -> UTF-8 bytes.
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

# A unique ID for this test message, so we can find this exact message
# again later (e.g. when a consumer reads it back from Kafka).
test_txn_id = str(uuid.uuid4())

# A fake UPI transaction, shaped like a real one, used only to verify
# the pipeline is wired up correctly end-to-end.
test_message = {
    "txn_id": test_txn_id,
    "sender_upi": "smoketest@oksbi",
    "receiver_upi": "merchant@okhdfcbank",
    "amount": 42.42,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}

producer.send(TOPIC, value=test_message)  # queue the message for sending (async)
producer.flush()  # block until the message is actually delivered to Kafka
print("sent:", test_message)

#the KafkaProducer in the Jupyter container connected to the Kafka broker's internal listener at kafka:29092 (the bootstrap address)
#  and published your test transaction onto the upi.transactions topic

sent: {'txn_id': '1aaebc9e-9999-4740-820c-f54767c7cfd0', 'sender_upi': 'smoketest@oksbi', 'receiver_upi': 'merchant@okhdfcbank', 'amount': 42.42, 'timestamp': '2026-07-19T10:38:16'}


In [6]:
from kafka import KafkaConsumer

# A consumer reads messages from a topic. auto_offset_reset="earliest" means:
# if this consumer group has no saved position yet, start reading from the
# very beginning of the topic (not just new messages from now on).
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    auto_offset_reset="earliest",
    enable_auto_commit=False,       # don't save read-progress; this is just a one-off test
    consumer_timeout_ms=10_000,     # stop waiting and exit the loop after 10s of no new messages
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),  # reverse of the producer's serializer: bytes -> UTF-8 string -> dict
)

found = False
# Iterating over the consumer yields each message in the topic, in order.
for record in consumer:
    # We check txn_id because the topic may contain older messages from
    # previous runs too — we only care about the one we just sent.
    if record.value.get("txn_id") == test_txn_id:
        found = True
        print("found test message:", record.value)
        break  # stop as soon as we find it, no need to keep reading

consumer.close()  # release the connection to the broker
assert found, "did not see the test message come back through Kafka"
print("Kafka round-trip OK")


/tmp/ipykernel_386/492243029.py:6: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


found test message: {'txn_id': '1aaebc9e-9999-4740-820c-f54767c7cfd0', 'sender_upi': 'smoketest@oksbi', 'receiver_upi': 'merchant@okhdfcbank', 'amount': 42.42, 'timestamp': '2026-07-19T10:38:16'}
Kafka round-trip OK


In [7]:
import os
import psycopg2

# credentials come from the container's env (passed in via docker-compose.yml,
# sourced from .env) instead of being hardcoded here
PG_CONN = dict(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.environ["POSTGRES_PORT"]),
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
)

conn = psycopg2.connect(**PG_CONN)
conn.autocommit = True
cur = conn.cursor()

cur.execute(
    """
    INSERT INTO fraud_alerts
        (txn_id, sender_upi, receiver_upi, amount, txn_timestamp, window_start, window_end, fraud_reason)
    VALUES (%s, %s, %s, %s, now(), now(), now(), %s)
    """,
    (test_txn_id, "smoketest@oksbi", "merchant@okhdfcbank", 42.42, "smoke_test"),
)
print("inserted test row")


inserted test row


In [8]:
cur.execute("SELECT txn_id, sender_upi, amount, fraud_reason FROM fraud_alerts WHERE txn_id = %s", (test_txn_id,))
row = cur.fetchone()
print("read back:", row)
assert row is not None, "test row was not found in fraud_alerts"

cur.execute("DELETE FROM fraud_alerts WHERE txn_id = %s", (test_txn_id,))
cur.close()
conn.close()
print("Postgres round-trip OK, test row cleaned up")


read back: ('1aaebc9e-9999-4740-820c-f54767c7cfd0', 'smoketest@oksbi', Decimal('42.42'), 'smoke_test')
Postgres round-trip OK, test row cleaned up


In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("smoke-test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.postgresql:postgresql:42.7.3",
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark session ready:", spark.version)


Spark session ready: 3.5.0


In [10]:
# spark.read (not readStream) means this is a one-time batch read, not a
# continuous streaming query — it loads what's currently on the topic and stops.
raw_df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC)          # which topic to read from
    .option("startingOffsets", "earliest")  # read from the beginning of the topic, not just new messages
    .load()
)

# Each row here is one Kafka record, with columns like key, value, topic,
# partition, offset, timestamp. count() forces Spark to actually read
# every message to compute the total.
print("messages on topic:", raw_df.count())

# 'value' comes back as raw bytes, so we cast it to a string to see the
# JSON payload we sent from the producer. show(5) prints up to 5 rows.
raw_df.selectExpr("CAST(value AS STRING) AS json_value").show(5, truncate=False)


messages on topic: 1
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json_value                                                                                                                                                                     |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"txn_id": "1aaebc9e-9999-4740-820c-f54767c7cfd0", "sender_upi": "smoketest@oksbi", "receiver_upi": "merchant@okhdfcbank", "amount": 42.42, "timestamp": "2026-07-19T10:38:16"}|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [11]:
jdbc_url = "jdbc:postgresql://postgres:5432/fraud_db"
jdbc_props = {"user": "fraud_user", "password": "fraud_pass", "driver": "org.postgresql.Driver"}

spark_test_df = spark.createDataFrame([(str(uuid.uuid4()), "spark-smoke-test")], ["txn_id", "note"])
spark_test_df.write.jdbc(url=jdbc_url, table="smoke_test_scratch", mode="overwrite", properties=jdbc_props)

readback_df = spark.read.jdbc(url=jdbc_url, table="smoke_test_scratch", properties=jdbc_props)
readback_df.show()
assert readback_df.count() == 1
print("Spark <-> Postgres JDBC OK")


+--------------------+----------------+
|              txn_id|            note|
+--------------------+----------------+
|179625ca-fc71-447...|spark-smoke-test|
+--------------------+----------------+

Spark <-> Postgres JDBC OK


In [12]:
conn = psycopg2.connect(**PG_CONN)
conn.autocommit = True
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS smoke_test_scratch")
cur.close()
conn.close()

spark.stop()
print("cleaned up")


cleaned up
